# Lecture 10: CNN可视化与理解 (CNN Visualization and Understanding)本笔记探索如何"打开黑箱"理解卷积神经网络：显著性图、Grad-CAM、对抗样本、特征反演和t-SNE嵌入。**学习目标：**- 理解CNN特征可视化的多种技术- 实现显著性图和Grad-CAM- 构造对抗样本并理解其原理- 实现特征反演和t-SNE降维- 分析网络学到的内部表示**四步教学路径：** 直觉理解 -> 手动计算 -> 代码实现 -> 实验观察

## 目录1. CNN可视化概述2. 显著性图 (Saliency Maps)3. Grad-CAM：梯度加权类激活映射4. 对抗样本 (Adversarial Examples)5. 特征反演 (Feature Inversion)6. t-SNE嵌入可视化7. 卷积核可视化8. 作业与参考文献

## 1. CNN可视化概述### 为什么需要可视化？深度神经网络常被称为"黑箱"。可视化技术帮助我们：1. **理解决策依据**：网络关注图像的哪些区域？2. **调试模型**：是否学到了正确的特征？3. **发现弱点**：对抗样本揭示了什么？4. **科学理解**：不同层的特征有什么区别？### 可视化技术分类| 类别 | 方法 | 回答的问题 ||---|---|---|| 输入归因 | Saliency, Grad-CAM | 这个像素对输出有多大贡献？ || 特征可视化 | 特征反演, 深度梦 | 什么样的输入能最大化这个特征？ || 表示分析 | t-SNE, PCA | 不同类别在特征空间中如何分布？ || 卷积核 | 滤波器可视化 | 这个卷积核检测什么模式？ || 对抗分析 | FGSM, DeepFool | 最小的扰动如何改变预测？ |

In [4]:
# -*- coding: utf-8 -*-import numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltnp.random.seed(42)# ============================================================# Simple CNN implementation (conv + relu + pool + fc)# ============================================================def conv2d(x, w, b=None, stride=1, padding=0):    # 2D convolution (actually cross-correlation)    if padding > 0:        x = np.pad(x, ((0,0),(padding,padding),(padding,padding)), mode='constant')    C, H, W = x.shape    F, _, KH, KW = w.shape    OH = (H - KH) // stride + 1    OW = (W - KW) // stride + 1    out = np.zeros((F, OH, OW))    for f in range(F):        for i in range(OH):            for j in range(OW):                region = x[:, i*stride:i*stride+KH, j*stride:j*stride+KW]                out[f, i, j] = np.sum(region * w[f]) + (b[f] if b is not None else 0)    return outdef relu(x):    return np.maximum(0, x)def maxpool2d(x, size=2):    C, H, W = x.shape    OH, OW = H // size, W // size    out = np.zeros((C, OH, OW))    for c in range(C):        for i in range(OH):            for j in range(OW):                out[c, i, j] = np.max(x[c, i*size:i*size+size, j*size:j*size+size])    return outdef global_avg_pool(x):    # Global average pooling    return x.mean(axis=(1, 2))print("CNN utilities defined: conv2d, relu, maxpool2d, global_avg_pool")

In [5]:
# ============================================================# Create a simple synthetic image and CNN model# ============================================================np.random.seed(42)# Create synthetic image (like a simple pattern: 32x32x3)img = np.zeros((3, 32, 32))# Draw a "square" patternimg[0, 8:24, 8:24] = 0.8   # Red channel has a squareimg[1, 10:22, 10:22] = 0.5 # Green channelimg[2, 12:20, 12:20] = 0.3 # Blue channel# Add some background noiseimg += np.random.randn(3, 32, 32) * 0.05img = np.clip(img, 0, 1)# Create simple CNN: conv(3->8, 3x3) -> relu -> pool -> conv(8->16, 3x3) -> relu -> pool -> GAP -> fcW1 = np.random.randn(8, 3, 3, 3) * 0.1b1 = np.zeros(8)W2 = np.random.randn(16, 8, 3, 3) * 0.1b2 = np.zeros(16)W_fc = np.random.randn(16, 3) * 0.1  # 3 classesb_fc = np.zeros(3)def forward(x):    # Forward pass storing intermediates    c1 = conv2d(x, W1, b1, padding=1)  # 8x32x32    r1 = relu(c1)    p1 = maxpool2d(r1, 2)  # 8x16x16    c2 = conv2d(p1, W2, b2, padding=1)  # 16x16x16    r2 = relu(c2)    p2 = maxpool2d(r2, 2)  # 16x8x8    gap = global_avg_pool(p2)  # 16    logits = gap @ W_fc + b_fc  # 3    cache = {'img': x, 'c1': c1, 'r1': r1, 'p1': p1, 'c2': c2, 'r2': r2, 'p2': p2, 'gap': gap}    return logits, cachelogits, cache = forward(img)probs = np.exp(logits) / np.exp(logits).sum()print(f"Image shape: {img.shape}")print(f"Logits: {logits}")print(f"Probabilities: {probs}")print(f"Predicted class: {probs.argmax()}")

## 2. 显著性图 (Saliency Maps)### 直觉显著性图回答一个简单的问题：**每个输入像素对网络输出的影响有多大？**通过计算输出对输入的梯度，我们得到每个像素的"重要性"。### 手动计算对于输入 x 和输出 y = f(x)：1. 计算梯度：dy/dx（输出对输入的偏导）2. 取绝对值（或平方）3. 沿通道维度取最大值（对RGB图像）4. 归一化到[0, 1]**数值示例：**- 3x3灰度图像，梯度 g = [[0.1, 0.5, 0.2], [0.3, 0.9, 0.4], [0.0, 0.1, 0.05]]- 取绝对值后不变（已为正）- 归一化：g / max(g) = [[0.11, 0.56, 0.22], [0.33, 1.0, 0.44], [0, 0.11, 0.06]]- 中心像素最重要！

In [7]:
# ============================================================# Saliency Map via numerical gradient# ============================================================def compute_saliency(img, target_class):    # Compute saliency using numerical gradient    h = 1e-3    grad = np.zeros_like(img)        # For each channel and pixel, compute d(output)/d(pixel)    for c in range(img.shape[0]):        for i in range(img.shape[1]):            for j in range(img.shape[2]):                old = img[c, i, j]                img[c, i, j] = old + h                logits_plus, _ = forward(img)                img[c, i, j] = old - h                logits_minus, _ = forward(img)                img[c, i, j] = old                grad[c, i, j] = (logits_plus[target_class] - logits_minus[target_class]) / (2 * h)        # Take max absolute value across channels    saliency = np.max(np.abs(grad), axis=0)    # Normalize    saliency = saliency / (saliency.max() + 1e-8)    return saliency# Compute saliency for the predicted classtarget = probs.argmax()saliency = compute_saliency(img.copy(), target)fig, axes = plt.subplots(1, 2, figsize=(10, 4))# Original image (transpose to HWC for display)axes[0].imshow(np.transpose(img, (1, 2, 0)))axes[0].set_title('Original Image', fontsize=12)axes[0].axis('off')# Saliency mapaxes[1].imshow(saliency, cmap='hot')axes[1].set_title(f'Saliency Map (class={target})', fontsize=12)axes[1].axis('off')plt.suptitle('Saliency Map Visualization', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/saliency.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 1 saved: saliency.png")

## 3. Grad-CAM：梯度加权类激活映射### 直觉Grad-CAM利用最后一个卷积层的梯度，生成粗粒度的类激活热力图。它回答：**网络在做这个分类决策时，关注了图像的哪个区域？**### 手动计算1. 前向传播，获取最后一层卷积特征 A (C x H x W) 和类别分数 y2. 计算梯度 dy/dA（类别分数对特征的梯度）3. 全局平均池化梯度得到权重 alpha = GAP(dy/dA) (C维向量)4. 加权求和：M = ReLU(sum_c(alpha_c * A_c))**数值示例：**- 特征 A 有2个通道，2x2空间- A_0 = [[1, 2], [3, 4]], A_1 = [[0.5, 1], [1.5, 2]]- 梯度均值: alpha_0 = 0.8, alpha_1 = 0.2- 加权和: 0.8*[[1,2],[3,4]] + 0.2*[[0.5,1],[1.5,2]] = [[0.9, 1.8], [2.7, 3.6]]- ReLU: 不变（已为正）- 上采样到输入尺寸得到热力图

In [9]:
# ============================================================# Grad-CAM Implementation# ============================================================def grad_cam(img, target_class, cache):    # Use last conv layer features (c2)    features = cache['c2']  # (F, H, W)        # Compute gradient of target class score w.r.t. features    h = 1e-3    grad = np.zeros_like(features)        for f in range(features.shape[0]):        for i in range(features.shape[1]):            for j in range(features.shape[2]):                old = features[f, i, j]                # We need to perturb the feature and see effect on output                # Since we can't directly modify features, we use the stored cache                # Approximation: use the gradient of GAP w.r.t. features                pass        # Simplified: use weight from GAP path    # For our network: logits = GAP(p2) @ W_fc    # d(logits[target])/d(GAP) = W_fc[:, target]    # GAP = mean(p2) over spatial dims    # So alpha_c = W_fc[c, target] / (H*W)        p2 = cache['p2']  # (16, 8, 8)    H_feat, W_feat = p2.shape[1], p2.shape[2]        # Channel weights from FC layer    alpha = W_fc[:, target_class]  # (16,)        # Weighted combination    gradcam = np.zeros((H_feat, W_feat))    for c in range(p2.shape[0]):        gradcam += alpha[c] * p2[c]        gradcam = np.maximum(gradcam, 0)  # ReLU    gradcam = gradcam / (gradcam.max() + 1e-8)        return gradcam# Compute Grad-CAMgradcam = grad_cam(img, target, cache)# Upsample to input sizegradcam_upsampled = np.kron(gradcam, np.ones((4, 4)))[:32, :32]fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(np.transpose(img, (1, 2, 0)))axes[0].set_title('Original', fontsize=12)axes[0].axis('off')axes[1].imshow(gradcam, cmap='jet')axes[1].set_title('Grad-CAM (feature space)', fontsize=12)axes[1].axis('off')axes[2].imshow(np.transpose(img, (1, 2, 0)))axes[2].imshow(gradcam_upsampled, cmap='jet', alpha=0.5)axes[2].set_title('Grad-CAM Overlay', fontsize=12)axes[2].axis('off')plt.suptitle('Grad-CAM Visualization', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/gradcam.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 2 saved: gradcam.png")

## 4. 对抗样本 (Adversarial Examples)### 直觉对抗样本是对输入添加精心设计的小扰动，使网络做出错误预测。最著名的方法是FGSM（Fast Gradient Sign Method）：x_adv = x + eps * sign(grad_x(loss))### 手动计算**FGSM步骤：**1. 计算损失对输入的梯度 g = dL/dx2. 取符号：sign(g)（每个元素为+1或-1）3. 添加扰动：x_adv = x + eps * sign(g)**数值示例：**- 输入 x = [0.5, 0.3, 0.8]- 梯度 g = [0.01, -0.03, 0.02]- sign(g) = [1, -1, 1]- eps = 0.1- x_adv = [0.5+0.1, 0.3-0.1, 0.8+0.1] = [0.6, 0.2, 0.9]- 扰动很小，但可能改变预测！### 为什么对抗样本有效？1. **高维空间**：在高维空间中，很小的逐维扰动累积效果很大2. **线性特性**：W^T * (x + delta) = W^T*x + W^T*delta，delta虽小但W^T*delta可不小3. **决策边界**：网络学到的决策边界可能过于"尖锐"

In [11]:
# ============================================================# FGSM Adversarial Attack# ============================================================def fgsm_attack(img, target_class, eps=0.1):    # Compute gradient of loss w.r.t. input    h = 1e-3    grad = np.zeros_like(img)        for c in range(img.shape[0]):        for i in range(img.shape[1]):            for j in range(img.shape[2]):                old = img[c, i, j]                img[c, i, j] = old + h                logits_plus, _ = forward(img)                probs_plus = np.exp(logits_plus) / np.exp(logits_plus).sum()                loss_plus = -np.log(probs_plus[target_class] + 1e-12)                                img[c, i, j] = old - h                logits_minus, _ = forward(img)                probs_minus = np.exp(logits_minus) / np.exp(logits_minus).sum()                loss_minus = -np.log(probs_minus[target_class] + 1e-12)                                img[c, i, j] = old                # Gradient of loss (we want to increase loss to break prediction)                grad[c, i, j] = (loss_plus - loss_minus) / (2 * h)        # FGSM: add sign of gradient    adv_img = img + eps * np.sign(grad)    adv_img = np.clip(adv_img, 0, 1)    return adv_img, grad# Generate adversarial exampleorig_logits, _ = forward(img)orig_probs = np.exp(orig_logits) / np.exp(orig_logits).sum()orig_pred = orig_probs.argmax()adv_img, attack_grad = fgsm_attack(img.copy(), orig_pred, eps=0.15)adv_logits, _ = forward(adv_img)adv_probs = np.exp(adv_logits) / np.exp(adv_logits).sum()adv_pred = adv_probs.argmax()perturbation = adv_img - imgprint("=== FGSM Adversarial Attack ===")print(f"Original prediction: class {orig_pred} (prob={orig_probs[orig_pred]:.4f})")print(f"Adversarial prediction: class {adv_pred} (prob={adv_probs[adv_pred]:.4f})")print(f"Perturbation L2 norm: {np.linalg.norm(perturbation):.4f}")print(f"Max pixel perturbation: {np.abs(perturbation).max():.4f}")print(f"Prediction changed: {orig_pred != adv_pred}")

In [12]:
# ============================================================# Visualize adversarial example# ============================================================fig, axes = plt.subplots(1, 3, figsize=(15, 4))# Originalaxes[0].imshow(np.transpose(img, (1, 2, 0)))axes[0].set_title(f'Original\nClass {orig_pred} ({orig_probs[orig_pred]:.3f})', fontsize=11)axes[0].axis('off')# Perturbation (amplified)pert_vis = perturbation * 10  # Amplify for visibilityaxes[1].imshow(np.transpose(pert_vis, (1, 2, 0)))axes[1].set_title('Perturbation (x10)', fontsize=11)axes[1].axis('off')# Adversarialaxes[2].imshow(np.transpose(adv_img, (1, 2, 0)))axes[2].set_title(f'Adversarial\nClass {adv_pred} ({adv_probs[adv_pred]:.3f})', fontsize=11)axes[2].axis('off')plt.suptitle('FGSM Adversarial Example', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/adversarial.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 3 saved: adversarial.png")

In [13]:
# ============================================================# Experiment: Effect of epsilon on attack success# ============================================================epsilons = [0, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0]attack_results = []for eps in epsilons:    adv, _ = fgsm_attack(img.copy(), orig_pred, eps=eps)    adv_l, _ = forward(adv)    adv_p = np.exp(adv_l) / np.exp(adv_l).sum()    attack_results.append({        'eps': eps,        'pred': adv_p.argmax(),        'conf': adv_p[adv_p.argmax()],        'orig_conf': adv_p[orig_pred]    })fig, ax = plt.subplots(figsize=(10, 5))ax.plot(epsilons, [r['orig_conf'] for r in attack_results], 'bo-', linewidth=2, label='Original class prob')ax.plot(epsilons, [r['conf'] for r in attack_results], 'rs-', linewidth=2, label='Adversarial class prob')ax.set_xlabel('Epsilon (perturbation size)')ax.set_ylabel('Probability')ax.set_title('FGSM: Effect of Epsilon on Attack', fontsize=14, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/fgsm_eps.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 4 saved: fgsm_eps.png")for r in attack_results:    print(f"eps={r['eps']:.2f}: pred={r['pred']}, orig_conf={r['orig_conf']:.4f}")

## 5. 特征反演 (Feature Inversion)### 直觉特征反演从中间特征重建输入图像，回答：**这个特征保留了多少输入信息？**### 算法1. 选择一个中间层特征2. 从随机噪声开始3. 通过梯度下降优化：找到使特征最接近目标特征的输入min_x ||f(x) - f(x0)||^2其中 f(x0) 是目标图像的特征。### 手动计算目标特征: F_target = [0.5, 0.3]当前特征: F_current = [0.1, 0.2]损失: L = (0.5-0.1)^2 + (0.3-0.2)^2 = 0.16 + 0.01 = 0.17梯度方向: 朝着减小损失的方向调整 x### 不同层的特征反演- 浅层特征保留了更多空间细节- 深层特征保留了高层语义但丢失纹理

In [15]:
# ============================================================# Feature Inversion (simplified)# ============================================================def feature_inversion(target_features, layer_name, n_iter=50, lr=0.5):    # Start from random noise    x = np.random.rand(3, 32, 32) * 0.1        losses = []    for it in range(n_iter):        # Forward pass        logits, cache = forward(x)        current_features = cache[layer_name]                # L2 loss between features        diff = current_features - target_features        loss = np.sum(diff ** 2)        losses.append(loss)                # Numerical gradient w.r.t. input (simplified)        # Use perturbation to estimate gradient direction        grad = np.zeros_like(x)        h = 0.01        for c in range(3):            for i in range(0, 32, 4):  # Subsample for speed                for j in range(0, 32, 4):                    old = x[c, i, j]                    x[c, i, j] = old + h                    _, c2 = forward(x)                    loss_plus = np.sum((c2[layer_name] - target_features) ** 2)                    x[c, i, j] = old - h                    _, c3 = forward(x)                    loss_minus = np.sum((c3[layer_name] - target_features) ** 2)                    x[c, i, j] = old                    grad[c, i, j] = (loss_plus - loss_minus) / (2 * h)                x -= lr * grad        x = np.clip(x, 0, 1)        return x, losses# Get target features from original image_, target_cache = forward(img)target_feat = target_cache['p2'].copy()# Run feature inversion (reduced iterations for speed)reconstructed, inv_losses = feature_inversion(target_feat, 'p2', n_iter=30, lr=0.3)fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(np.transpose(img, (1, 2, 0)))axes[0].set_title('Original', fontsize=12)axes[0].axis('off')axes[1].imshow(np.transpose(reconstructed, (1, 2, 0)))axes[1].set_title('Reconstructed from p2 features', fontsize=12)axes[1].axis('off')axes[2].plot(inv_losses, linewidth=2)axes[2].set_xlabel('Iteration')axes[2].set_ylabel('Feature L2 Loss')axes[2].set_title('Inversion Loss', fontsize=12)axes[2].grid(True, alpha=0.3)plt.suptitle('Feature Inversion', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/feature_inversion.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 5 saved: feature_inversion.png")print(f"Final inversion loss: {inv_losses[-1]:.6f}")

## 6. t-SNE嵌入可视化### 直觉t-SNE将高维特征映射到2D/3D空间，保持局部邻域结构。相似的样本在2D空间中也会靠近，不相似的样本会远离。### 算法核心步骤1. **计算高维相似度**：P(j|i) = exp(-||x_i - x_j||^2 / 2*sigma^2) / sum_k(exp(...))2. **初始化低维嵌入**：随机初始化 y_i3. **计算低维相似度**：Q(j|i) = (1 + ||y_i - y_j||^2)^(-1) / sum_k(...)4. **最小化KL散度**：KL(P||Q) = sum(P * log(P/Q))5. **梯度下降**更新 y_i### 手动计算（简化版）3个点的高维特征：- A = [1, 0], B = [0, 1], C = [5, 5]- A和B相似，C远离两者- 目标：在2D中保持A-B近，C远高维距离：d(A,B) = sqrt(2), d(A,C) = sqrt(41), d(B,C) = sqrt(41)相似度 P(B|A) = exp(-2/2) = 0.37t-SNE会在低维中也让A-B比A-C更近。

In [17]:
# ============================================================# Simple t-SNE Implementation# ============================================================def simple_tsne(X, n_components=2, perplexity=5, n_iter=300, lr=100):    # Simplified t-SNE for small datasets    N = X.shape[0]        # Compute pairwise distances    sum_X = np.sum(X**2, axis=1)    dists = np.abs(np.add.outer(sum_X, sum_X) - 2 * X @ X.T)        # Compute P (high-dimensional similarities)    P = np.exp(-dists / (2 * np.mean(dists)))    np.fill_diagonal(P, 0)    P = P / P.sum()    P = np.maximum(P, 1e-12)        # Initialize low-dimensional embedding    Y = np.random.randn(N, n_components) * 1e-4        # Gradient descent    for it in range(n_iter):        # Compute Q (low-dimensional similarities)        sum_Y = np.sum(Y**2, axis=1)        dists_Y = np.abs(np.add.outer(sum_Y, sum_Y) - 2 * Y @ Y.T)        Q = (1 + dists_Y) ** (-1)        np.fill_diagonal(Q, 0)        Q = Q / Q.sum()        Q = np.maximum(Q, 1e-12)            # Gradient of KL divergence        PQ = (P - Q) * Q        grad = np.zeros_like(Y)        for i in range(N):            grad[i] = 4 * np.sum(np.outer(PQ[i], Y[i] - Y), axis=0)                # Momentum        if it == 0:            dY = grad.copy()        else:            dY = 0.5 * dY - lr * grad                Y += dY        Y -= Y.mean(0)  # Center        return Y# Generate synthetic feature data (3 clusters in high-D space)np.random.seed(42)n_per_class = 30features = []labels = []centers = [np.random.randn(10) * 3, np.random.randn(10) * 3, np.random.randn(10) * 3]for c in range(3):    features.append(centers[c] + np.random.randn(n_per_class, 10) * 0.5)    labels.extend([c] * n_per_class)features = np.vstack(features)labels = np.array(labels)print(f"Feature data: {features.shape}, {len(labels)} labels")print("Running t-SNE...")embedding = simple_tsne(features, n_iter=300, lr=50)print(f"Embedding shape: {embedding.shape}")

In [18]:
# ============================================================# Visualize t-SNE embedding# ============================================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# PCA for comparisonfrom numpy.linalg import svdX_centered = features - features.mean(0)U, S, Vt = svd(X_centered, full_matrices=False)pca_embedding = X_centered @ Vt[:2].Tcolors = ['red', 'blue', 'green']for c in range(3):    mask = labels == c    axes[0].scatter(pca_embedding[mask, 0], pca_embedding[mask, 1], c=colors[c], label=f'Class {c}', alpha=0.7, s=30)axes[0].set_title('PCA (2D)', fontsize=13, fontweight='bold')axes[0].legend()axes[0].grid(True, alpha=0.3)for c in range(3):    mask = labels == c    axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=colors[c], label=f'Class {c}', alpha=0.7, s=30)axes[1].set_title('t-SNE (2D)', fontsize=13, fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.suptitle('Feature Space Visualization: PCA vs t-SNE', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/tsne.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 6 saved: tsne.png")

## 7. 卷积核可视化### 直觉每个卷积核学习检测特定模式。我们可以直接将卷积核权重可视化来理解它检测什么。### 手动计算一个3x3x3的卷积核可以看作3个通道的3x3矩阵：- 如果某个通道的权重有明显的方向性（如水平边缘检测器），说明该核在检测对应方向的边缘- 常见模式：边缘检测器、颜色块检测器、纹理检测器**经典滤波器：**- Sobel (水平): [[-1,0,1],[-2,0,2],[-1,0,1]]- Sobel (垂直): [[-1,-2,-1],[0,0,0],[1,2,1]]- Gaussian: [[1,2,1],[2,4,2],[1,2,1]] / 16

In [20]:
# ============================================================# Convolutional filter visualization# ============================================================# Visualize first layer filtersfig, axes = plt.subplots(2, 4, figsize=(12, 6))axes = axes.flatten()for i in range(W1.shape[0]):    # Normalize filter for display    f = W1[i]  # (3, 3, 3)    f = f - f.min()    f = f / (f.max() + 1e-8)    axes[i].imshow(np.transpose(f, (1, 2, 0)))    axes[i].set_title(f'Filter {i}', fontsize=10)    axes[i].axis('off')plt.suptitle('First Layer Convolutional Filters', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/filters.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 7 saved: filters.png")

In [21]:
# ============================================================# Feature map visualization# ============================================================fig, axes = plt.subplots(2, 4, figsize=(12, 6))axes = axes.flatten()for i in range(8):    axes[i].imshow(cache['r1'][i], cmap='viridis')    axes[i].set_title(f'Feature {i}', fontsize=10)    axes[i].axis('off')plt.suptitle('First Layer Feature Maps (after ReLU)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/feature_maps.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 8 saved: feature_maps.png")

## 7.5 深度梦境 (Deep Dream) 原理### 直觉深度梦境通过**放大**网络检测到的特征来生成图像：x_new = x + lr * grad(f(x))不同于特征反演（最小化特征差异），深度梦境是**最大化**某些特征的激活。### 手动计算1. 选择一个中间层2. 计算该层激活的总和作为"目标"3. 对输入求梯度4. 将梯度直接加到输入上（放大模式）5. 重复几次**关键区别：**- 特征反演：让特征匹配特定目标（减小差异）- 深度梦境：让特征尽可能大（增强模式）- 对抗样本：让输出改变（最大化损失）

In [23]:
# ============================================================# Simple Deep Dream (amplify features)# ============================================================def deep_dream(img, layer_name, n_iter=20, lr=0.05):    x = img.copy()    activations = []        for it in range(n_iter):        _, cache = forward(x)        feat = cache[layer_name]        # Objective: maximize sum of activations        obj = np.sum(feat)        activations.append(obj)                # Numerical gradient (subsampled for speed)        grad = np.zeros_like(x)        h = 0.01        for c in range(3):            for i in range(0, 32, 4):                for j in range(0, 32, 4):                    old = x[c, i, j]                    x[c, i, j] = old + h                    _, c2 = forward(x)                    obj_plus = np.sum(c2[layer_name])                    x[c, i, j] = old - h                    _, c3 = forward(x)                    obj_minus = np.sum(c3[layer_name])                    x[c, i, j] = old                    grad[c, i, j] = (obj_plus - obj_minus) / (2 * h)                # Amplify features        x += lr * grad        x = np.clip(x, 0, 1)        return x, activationsdream_img, dream_acts = deep_dream(img.copy(), 'r2', n_iter=15, lr=0.03)fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(np.transpose(img, (1, 2, 0)))axes[0].set_title('Original', fontsize=12)axes[0].axis('off')axes[1].imshow(np.transpose(dream_img, (1, 2, 0)))axes[1].set_title('Deep Dream (amplified r2)', fontsize=12)axes[1].axis('off')axes[2].plot(dream_acts, 'go-', linewidth=2)axes[2].set_xlabel('Iteration')axes[2].set_ylabel('Feature Activation Sum')axes[2].set_title('Activation Growth', fontsize=12)axes[2].grid(True, alpha=0.3)plt.suptitle('Deep Dream Visualization', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/deep_dream.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 9 saved: deep_dream.png")

## 8. 综合对比：不同层的特征分析### 不同层特征的差异| 层 | 特征类型 | t-SNE分布 | 特征反演质量 ||---|---|---|---|| 浅层(conv1) | 边缘、颜色 | 类别重叠多 | 接近原图 || 中层(conv2) | 纹理、部件 | 类别开始分离 | 保留结构 || 深层(GAP) | 语义、对象 | 类别明显分离 | 抽象表示 |### 可视化方法对比| 方法 | 优点 | 缺点 ||---|---|---|| Saliency | 像素级精度 | 可能噪声大 || Grad-CAM | 粗粒度定位 | 分辨率低 || t-SNE | 全局结构 | 计算量大 || 特征反演 | 直观理解 | 优化困难 || 对抗样本 | 发现弱点 | 可能有安全性问题 |

In [25]:
# ============================================================# Compare features across layers using t-SNE# ============================================================# Generate multiple synthetic images per classnp.random.seed(42)n_samples = 15all_features_conv1 = []all_features_conv2 = []all_labels = []for c in range(3):    for _ in range(n_samples):        # Create variant of base pattern        x = np.zeros((3, 32, 32))        offset = np.random.randint(-4, 5)        size_var = np.random.randint(10, 20)        x[0, 8+offset:8+offset+size_var, 8+offset:8+offset+size_var] = 0.5 + np.random.rand() * 0.3        x[1, 10+offset:10+offset+size_var-2, 10+offset:10+offset+size_var-2] = 0.3 + np.random.rand() * 0.2        x += np.random.randn(3, 32, 32) * 0.05        x = np.clip(x, 0, 1)                _, c_fwd = forward(x)        all_features_conv1.append(global_avg_pool(c_fwd['r1']))        all_features_conv2.append(global_avg_pool(c_fwd['r2']))        all_labels.append(c)feats1 = np.array(all_features_conv1)feats2 = np.array(all_features_conv2)labels_arr = np.array(all_labels)# Run t-SNE on each layeremb1 = simple_tsne(feats1, n_iter=200, lr=50)emb2 = simple_tsne(feats2, n_iter=200, lr=50)fig, axes = plt.subplots(1, 2, figsize=(12, 5))for c in range(3):    m = labels_arr == c    axes[0].scatter(emb1[m, 0], emb1[m, 1], c=colors[c], label=f'Class {c}', alpha=0.7, s=30)    axes[1].scatter(emb2[m, 0], emb2[m, 1], c=colors[c], label=f'Class {c}', alpha=0.7, s=30)axes[0].set_title('Layer 1 Features (t-SNE)', fontweight='bold')axes[1].set_title('Layer 2 Features (t-SNE)', fontweight='bold')for ax in axes:    ax.legend()    ax.grid(True, alpha=0.3)plt.suptitle('Feature Separation Across Layers', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/layer_tsne.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 10 saved: layer_tsne.png")

## 8.5 遮挡敏感性分析 (Occlusion Sensitivity)### 直觉遮挡敏感性通过系统地遮挡图像的不同区域，观察预测概率的变化，回答：**哪些区域对分类决策最关键？**### 算法1. 选择一个滑动窗口大小（如 8x8）2. 在图像上滑动，将窗口区域置零（或灰色）3. 记录每次遮挡后的分类概率4. 概率下降最多的区域 = 关键区域### 与Saliency对比| 方法 | 优势 | 劣势 ||---|---|---|| Saliency | 精确到像素 | 可能噪声大 || Occlusion | 直观可靠 | 计算量大 || Grad-CAM | 平衡精确度和速度 | 分辨率低 |

In [27]:
# ============================================================# Occlusion Sensitivity Analysis# ============================================================def occlusion_sensitivity(img, target_class, occ_size=8, occ_stride=8):    H, W = img.shape[1], img.shape[2]    sensitivity = np.zeros((H, W))        for i in range(0, H, occ_stride):        for j in range(0, W, occ_stride):            x_occ = img.copy()            # Occlude region with gray            i_end = min(i + occ_size, H)            j_end = min(j + occ_size, W)            x_occ[:, i:i_end, j:j_end] = 0.5                        logits, _ = forward(x_occ)            probs = np.exp(logits) / np.exp(logits).sum()            sensitivity[i:i_end, j:j_end] = probs[target_class]        return sensitivity# Compute occlusion sensitivityocc_map = occlusion_sensitivity(img.copy(), target, occ_size=8, occ_stride=4)fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].imshow(np.transpose(img, (1, 2, 0)))axes[0].set_title('Original', fontsize=12)axes[0].axis('off')axes[1].imshow(occ_map, cmap='RdYlGn')axes[1].set_title('Occlusion Sensitivity\n(green=high prob)', fontsize=11)axes[1].axis('off')axes[2].imshow(np.transpose(img, (1, 2, 0)))prob_drop = np.maximum(0, occ_map.max() - occ_map)axes[2].imshow(prob_drop, cmap='hot', alpha=0.6)axes[2].set_title('Probability Drop\n(hot=critical region)', fontsize=11)axes[2].axis('off')plt.suptitle('Occlusion Sensitivity Analysis', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/occlusion.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 11 saved: occlusion.png")

## 8.6 激活最大化 (Activation Maximization)### 直觉激活最大化寻找使某个神经元激活值最大的输入，回答：**什么样的输入最能激活这个神经元？**### 手动计算1. 选择一个神经元2. 初始化随机输入3. 计算该神经元的激活值4. 对输入求梯度，沿梯度上升方向更新5. 重复直到收敛x_new = x + lr * d(activation)/d(x)这与深度梦境类似，但针对特定神经元而非整个层。

In [29]:
# ============================================================# Activation Maximization for specific neurons# ============================================================def activation_maximization(neuron_idx, layer_name, n_iter=20, lr=0.05):    x = np.random.rand(3, 32, 32) * 0.1    activations = []        for it in range(n_iter):        _, cache = forward(x)        feat = cache[layer_name]        # Target: maximize activation of specific neuron at center        h, w = feat.shape[1] // 2, feat.shape[2] // 2        act_val = feat[neuron_idx, h, w]        activations.append(act_val)                # Numerical gradient        grad = np.zeros_like(x)        hh = 0.01        for c in range(3):            for i in range(0, 32, 4):                for j in range(0, 32, 4):                    old = x[c, i, j]                    x[c, i, j] = old + hh                    _, c2 = forward(x)                    act_plus = c2[layer_name][neuron_idx, h, w]                    x[c, i, j] = old - hh                    _, c3 = forward(x)                    act_minus = c3[layer_name][neuron_idx, h, w]                    x[c, i, j] = old                    grad[c, i, j] = (act_plus - act_minus) / (2 * hh)                x += lr * grad        x = np.clip(x, 0, 1)        return x, activations# Maximize activation of neuron 0 and neuron 5fig, axes = plt.subplots(2, 3, figsize=(15, 8))for row, n_idx in enumerate([0, 5]):    max_img, max_acts = activation_maximization(n_idx, 'r2', n_iter=15, lr=0.03)        axes[row, 0].imshow(np.transpose(max_img, (1, 2, 0)))    axes[row, 0].set_title(f'Max input for neuron {n_idx}', fontsize=11)    axes[row, 0].axis('off')        axes[row, 1].imshow(max_img[0], cmap='Reds')    axes[row, 1].set_title(f'Red channel (neuron {n_idx})', fontsize=11)    axes[row, 1].axis('off')        axes[row, 2].plot(max_acts, 'o-', linewidth=2)    axes[row, 2].set_title(f'Activation growth (neuron {n_idx})', fontsize=11)    axes[row, 2].set_xlabel('Iteration')    axes[row, 2].set_ylabel('Activation')    axes[row, 2].grid(True, alpha=0.3)plt.suptitle('Activation Maximization', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/act_max.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 12 saved: act_max.png")

## 8.7 对抗训练 (Adversarial Training)### 直觉对抗训练通过在训练时同时使用干净样本和对抗样本来提升模型鲁棒性：L_adv = L(x, y) + L(x_adv, y)其中 x_adv = x + eps * sign(grad_x L(x, y))### 手动计算训练步骤：1. 前向传播计算损失 L(x, y)2. 计算梯度 g = dL/dx3. 生成对抗样本 x_adv = x + eps * sign(g)4. 计算对抗损失 L(x_adv, y)5. 总损失 = L(x, y) + L(x_adv, y)6. 反向传播更新权重### 效果- 对抗训练后的模型对对抗样本更鲁棒- 但可能在干净样本上精度下降- 训练更慢（需要两次前向+反向传播）

In [31]:
# ============================================================# Compare clean vs adversarial prediction confidence# ============================================================np.random.seed(42)eps_values = np.linspace(0, 0.5, 20)clean_confs = []adv_confs = []for eps in eps_values:    adv, _ = fgsm_attack(img.copy(), target, eps=eps)    adv_l, _ = forward(adv)    adv_p = np.exp(adv_l) / np.exp(adv_l).sum()        orig_l, _ = forward(img)    orig_p = np.exp(orig_l) / np.exp(orig_l).sum()        clean_confs.append(orig_p[target])    adv_confs.append(adv_p[target])fig, ax = plt.subplots(figsize=(10, 5))ax.plot(eps_values, clean_confs, 'b-', linewidth=2, label='Clean image confidence')ax.plot(eps_values, adv_confs, 'r-', linewidth=2, label='Adversarial confidence')ax.fill_between(eps_values, adv_confs, clean_confs, alpha=0.2, color='red', label='Confidence gap')ax.set_xlabel('Epsilon (perturbation size)')ax.set_ylabel('Target class probability')ax.set_title('Adversarial Robustness Analysis', fontsize=14, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/robustness.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 13 saved: robustness.png")

## 8.8 可视化方法的选择指南### 什么时候用什么方法？**如果想知道"网络看了哪里"：**- 快速粗略 -> Grad-CAM- 精确到像素 -> Saliency / Guided Backprop- 可靠但慢 -> Occlusion Sensitivity**如果想理解特征语义：**- 单神经元 -> Activation Maximization- 整个层 -> Deep Dream- 重建输入 -> Feature Inversion**如果想分析数据分布：**- 高维到2D -> t-SNE（保持局部结构）- 高维到2D -> PCA（保持全局方差）**如果关心模型安全：**- 攻击 -> FGSM, PGD- 防御 -> Adversarial Training

In [33]:
# ============================================================# Summary: All visualization methods comparison# ============================================================methods = ['Saliency', 'Grad-CAM', 'Occlusion', 't-SNE', 'Feature Inv.', 'Deep Dream', 'Act. Max.', 'FGSM']resolution = ['Pixel', 'Coarse', 'Coarse', 'N/A', 'Pixel', 'Pixel', 'Pixel', 'Pixel']speed = ['Fast', 'Fast', 'Slow', 'Slow', 'Slow', 'Slow', 'Slow', 'Fast']purpose = ['Attribution', 'Attribution', 'Attribution', 'Analysis', 'Understand', 'Generate', 'Understand', 'Attack']fig, ax = plt.subplots(figsize=(10, 6))colors_map = {'Attribution': 'skyblue', 'Analysis': 'lightgreen', 'Understand': 'orange', 'Generate': 'pink', 'Attack': 'salmon'}for i, (m, r, s, p) in enumerate(zip(methods, resolution, speed, purpose)):    ax.barh(i, 1, color=colors_map[p], edgecolor='black', alpha=0.8)    ax.text(0.5, i, f'{m}\n({p}, {s}, {r})', ha='center', va='center', fontsize=9, fontweight='bold')ax.set_yticks(range(len(methods)))ax.set_yticklabels(['' for _ in methods])ax.set_xlim(0, 1)ax.set_title('CNN Visualization Methods Overview', fontsize=14, fontweight='bold')ax.set_xlabel('')legend_elements = [plt.Rectangle((0,0),1,1, color=c, label=l) for l, c in colors_map.items()]ax.legend(handles=legend_elements, loc='lower right', fontsize=10)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-10-visualization/methods_overview.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 14 saved: methods_overview.png")

## 9. 关键要点总结1. **Saliency Maps** 通过梯度识别重要像素，但可能噪声大2. **Grad-CAM** 提供粗粒度定位，更稳定但分辨率低3. **对抗样本** 揭示了深度网络的脆弱性，FGSM是最简单的攻击方法4. **特征反演** 从中间特征重建输入，展示信息保留程度5. **t-SNE** 是分析高维特征分布的强大工具6. **深层特征**更语义化，浅层特征更结构化### 安全启示对抗样本的存在意味着：- 深度学习模型不可在高风险场景中盲目部署- 需要对抗训练（adversarial training）增强鲁棒性- 对抗样本具有跨模型迁移性

## 10. 作业### 作业1：实现PGD (Projected Gradient Descent) 攻击PGD是FGSM的多步版本，更强但更慢：**算法：**1. 从原始图像出发2. 每步计算梯度，沿梯度方向走一小步3. 将扰动投影到eps球内（约束总扰动）4. 重复N步**要求：**1. 实现PGD攻击2. 对比FGSM和PGD在不同eps下的攻击成功率3. 分析PGD为何更强4. 绘制不同步数(5, 10, 20, 50)的效果对比

### 作业2：实现Guided BackpropagationGuided Backprop是Saliency的改进版本：**核心思想：**- 标准反向传播：梯度通过ReLU层- Guided Backprop：只允许正梯度通过正激活的ReLU**要求：**1. 实现Guided Backprop2. 与标准Saliency对比3. 分析为什么Guided Backprop更清晰4. 在多个类别上测试并比较

### 作业3：实现CBAM (Convolutional Block Attention Module)CBAM在通道和空间两个维度上添加注意力机制：**通道注意力：**- M_c(F) = sigmoid(MLP(GAP(F)) + MLP(GMP(F)))**空间注意力：**- M_s(F) = sigmoid(f([GAP(F); GMP(F)]))其中 GAP = 全局平均池化，GMP = 全局最大池化**要求：**1. 实现CBAM模块2. 将其插入CNN的不同位置3. 分析注意力图的变化4. 对比有无CBAM的分类性能

## 11. 参考文献1. [[Simonyan et al., 2014]](https://arxiv.org/abs/1312.6034) - Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps2. [[Springenberg et al., 2015]](https://arxiv.org/abs/1412.6806) - Striving for Simplicity: The All Convolutional Net (Guided Backpropagation)3. [[Selvaraju et al., 2017]](https://arxiv.org/abs/1610.02391) - Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization4. [[Goodfellow et al., 2015]](https://arxiv.org/abs/1412.6572) - Explaining and Harnessing Adversarial Examples (FGSM)5. [[Madry et al., 2018]](https://arxiv.org/abs/1706.06083) - Towards Deep Learning Models Resistant to Adversarial Attacks (PGD)6. [[Mahendran and Vedaldi, 2015]](https://arxiv.org/abs/1412.0035) - Understanding Deep Image Representations by Inverting Them (Feature Inversion)7. [[Maaten and Hinton, 2008]](https://www.jmlr.org/papers/v9/vandermaaten08a.html) - Visualizing Data using t-SNE8. [[Mordvintsev et al., 2015]](https://ai.googleblog.com/2015/06/inceptionism-going-deeper-into-neural.html) - Inceptionism: Going Deeper into Neural Networks (Deep Dream)9. [[Woo et al., 2018]](https://arxiv.org/abs/1807.06521) - CBAM: Convolutional Block Attention Module10. [[Olah et al., 2017]](https://distill.pub/2017/feature-visualization/) - Feature Visualization

---

## 参考代码实现

以下 GitHub 仓库提供了本节内容的完整代码实现，建议结合学习：

- **[utkuozbulak/pytorch-cnn-visualizations](https://github.com/utkuozbulak/pytorch-cnn-visualizations)** (8233 stars): CNN 可视化技术合集（Grad-CAM、显著性图、DeepDream）
  - 仓库地址: https://github.com/utkuozbulak/pytorch-cnn-visualizations

- **[kazuto1011/grad-cam-pytorch](https://github.com/kazuto1011/grad-cam-pytorch)** (804 stars): Grad-CAM PyTorch 复现
  - 仓库地址: https://github.com/kazuto1011/grad-cam-pytorch

- **[jacobgil/vit-explain](https://github.com/jacobgil/vit-explain)** (1099 stars): ViT 注意力可视化
  - 仓库地址: https://github.com/jacobgil/vit-explain


> 标注说明: 以上仓库按热度排序，优先推荐 stars 最多的实现。

